In [7]:
import pandas as pd
import random
import json
import pickle
import gzip
from tqdm import tqdm

In [37]:
ce_path = "/nfs/primary/SPLADE/splade_sans/rq3/outputs/strategy1_scores.json.pkl.gz"

def read_ce_score(ce_path: str):
    HFG_PREFIX = "poo"
    if ce_path.startswith(HFG_PREFIX):
        hf_name = ce_path.replace(HFG_PREFIX, "")
        _url = file_map[hf_name]
        dl_manager = DownloadManager()
        ce_path = dl_manager.download(_url)
    res = {}
    with gzip.open(ce_path, "rb") as f:
        data = pickle.load(f)
        for qid in tqdm(data, desc=f"Preprocessing data from {ce_path}"):
            res[str(qid)] = {str(did): data[qid][did] for did in data[qid]}
    return res

ce_score = read_ce_score(ce_path)

Preprocessing data from /nfs/primary/SPLADE/splade_sans/rq3/outputs/strategy1_scores.json.pkl.gz: 100%|██████████| 334059/334059 [00:05<00:00, 64958.65it/s]


In [27]:
qrels_path = "./data/training_qrels.json"

def read_qrels(qrels_path: str, rel_threshold=0):
    qid2pos = {}
    IRDS_PREFIX = "poo"
    if qrels_path.startswith(IRDS_PREFIX):
        irds_name = qrels_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for qrel in dataset.qrels_iter():
            if qrel.relevance > rel_threshold:
                qid, did = qrel.query_id, qrel.doc_id
                if not qid in qid2pos:
                    qid2pos[qid] = []
                qid2pos[qid].append(did)
    else:
        qrels = json.load(open(qrels_path, "r"))
        for qid in qrels:
            qid2pos[str(qid)] = [str(did) for did in qrels[qid]]
    return qid2pos

qrels = read_qrels(qrels_path)

In [29]:
queries_path = "./data/training_queries.tsv"

def read_queries(queries_path: str, text_fields=["text"]):
    IRDS_PREFIX = "poo"
    queries = []
    if queries_path.startswith(IRDS_PREFIX):
        irds_name = queries_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for query in tqdm(
            dataset.queries_iter(),
            desc=f"Loading queries from ir_datasets: {queries_path}",
        ):
            query_id = query.query_id
            texts = [getattr(query, field) for field in text_fields]
            text = " ".join(texts)
            queries.append((query_id, text))
    else:
        with open(queries_path, "r") as f:
            for line in tqdm(f, desc=f"Reading queries from {queries_path}"):
                query_id, query_text = line.strip().split("\t")
                queries.append((query_id, query_text))
    return queries
    
queries = read_queries(queries_path)

Reading queries from ./data/training_queries.tsv: 334070it [00:00, 1513526.59it/s]


In [43]:
collection_path = "./data/text.tsv"

def read_collection(collection_path: str, text_fields=["text"]):
    doc_dict = {}
    IRDS_PREFIX = "poo"
    HFG_PREFIX = "poo"
    if collection_path.startswith(IRDS_PREFIX):
        irds_name = collection_path.replace(IRDS_PREFIX, "")
        dataset = ir_datasets.load(irds_name)
        for doc in tqdm(
            dataset.docs_iter(),
            desc=f"Loading doc collection from ir_datasets: {irds_name}",
        ):
            doc_id = doc.doc_id
            texts = [getattr(doc, field) for field in text_fields]
            text = " ".join(texts)
            doc_dict[doc_id] = text
    elif collection_path.startswith(HFG_PREFIX):
        hfg_name = collection_path.replace(HFG_PREFIX, "")
        dataset = load_dataset(hfg_name)
        for row in tqdm(
            dataset["passage"],
            desc=f"Loading data from HuggingFace datasets: {hfg_name}",
        ):
            doc_dict[row["id"]] = row["text"]
    else:
        with open(collection_path, "r") as f:
            for line in tqdm(f, desc=f"Reading doc collection from {collection_path}"):
                doc_id, doc_text = line.strip().split("\t")
                doc_dict[doc_id] = doc_text
    return doc_dict


doc_dict = read_collection(collection_path)

Reading doc collection from ./data/text.tsv: 334134it [00:00, 633006.62it/s]


In [76]:
train_group_size  = 2
for idx in range(len(queries)):
    q_id, q_text = queries[idx]
    if q_id in qrels:
        doc1_id = random.choice(qrels[q_id])
    else:
        if q_id in ce_score:
            doc1_id = random.choice(list(ce_score[q_id].keys()))
        else:
            continue
    doc_list = [doc_dict[doc1_id]]
    score_list = [ce_score[q_id][doc1_id]]
    all_doc_ids = list(ce_score[q_id].keys())
    if len(all_doc_ids) < train_group_size - 1:
        neg_doc_ids = random.choices(all_doc_ids, k=train_group_size - 1)
    else:
        neg_doc_ids = random.sample(all_doc_ids, k=train_group_size - 1)
    doc_list.extend([doc_dict[doc_id] for doc_id in neg_doc_ids])
    score_list.extend([ce_score[q_id][doc_id] for doc_id in neg_doc_ids])

88001800
88001810
88001815
88001816
88001821
88001902
88001903
88001917
88003317
88003518
88003676
88003678
88003689
88004285
88005670
88006763
88006768
88006857
88007142
88008585
88008751
88008752
88010073
88010276
88010379
88010380
88011486
88011572
88012618
88012620
88012621
88012622
88012688
88012792
88012794
88012891
88012918
88012919
88012929
88012961
88013184
88014046
88014063
88014082
88014090
88014165
88014602
88014843
88014959
88015283
88015380
88017017
88017992
88018210
88019043
88020687
88020697
88020698
88020699
88020700
88020702
88020714
88020732
88020757
88020758
88020762
88020766
88020769
88020774
88020780
88020781
88020814
88020817
88020820
88020822
88020828
88020835
88021097
88021151
88021185
88021704
88021709
88021949
88022050
88023271
88023305
88026013
88026092
88026104
88026128
88026206
88026994
88028799
88032292
88032294
88032297
88032298
88032300
88033178
88033187
88034144
88034161
88035091
88035243
88035441
88035455
88036281
88036321
88037151
88037716
88037736
8